# Conjuntos (Sets)

* Un conjunto es una colección abstracta de objetos bien definidos.
* La invariante principal de esta estructura de datos es la unicidad estricta; carece por completo de elementos duplicados.
* La cardinalidad representa el número absoluto de elementos únicos almacenados, denotado como $|A|$.
* La implementación garantiza la correcta ejecución del álgebra de conjuntos:
  * Unión: $A \cup B = \{x \mid x \in A \lor x \in B\}$
  * Intersección: $A \cap B = \{x \mid x \in A \land x \in B\}$
  * Diferencia: $A \setminus B = \{x \mid x \in A \land x \notin B\}$
  * Subconjunto: $A \subseteq B \iff \forall x (x \in A \implies x \in B)$

## Diseño Contractual de la Interfaz Estricta

La arquitectura del sistema se fundamenta en el principio de Inversión de Dependencias mediante la definición de una clase abstracta base. Esta abstracción establece un contrato operativo que cualquier estructura de datos subyacente debe satisfacer obligatoriamente. La utilización de plantillas genéricas en C++ permite que el conjunto opere de forma agnóstica respecto al tipo de dato, delegando las responsabilidades de evaluación de igualdad lógica al tipo contenido.

In [1]:
#include <iostream>
#include <vector>
#include <algorithm>
#include <stdexcept>
#include <chrono>
#include <random>
#include <iomanip>
#include <cassert>

/**
 * @interface ISet
 * @brief Interfaz abstracta genérica para la estructura de datos Conjunto.
 * * Define el contrato estándar para las operaciones básicas de un conjunto matemático,
 * garantizando la ausencia de elementos duplicados.
 * * @tparam T Tipo de los elementos almacenados. Debe soportar el operador de igualdad (==).
 */
template <typename T>
class ISet {
protected:
    size_t size = 0; ///< Mantiene el conteo en caché para evitar llamadas a funciones dependientes del contenedor.

public:
    virtual ~ISet() = default;

    /**
     * @brief Obtiene la cardinalidad actual del conjunto.
     * @return Número de elementos almacenados.
     */
    virtual size_t getSize() const { return size; }

    /**
     * @brief Verifica si el conjunto carece de elementos.
     * @return true si el conjunto está vacío, false en caso contrario.
     */
    virtual bool isEmpty() const { return size == 0; }

    /** @brief Elimina todos los elementos del conjunto, restableciendo su tamaño a 0. */
    virtual void clear() = 0;

    /** @brief Imprime los elementos del conjunto en la salida estándar. */
    virtual void print() const = 0;

    /**
     * @brief Evalúa la pertenencia de un elemento al conjunto.
     * @param item Elemento a buscar.
     * @return true si el elemento existe, false en caso contrario.
     */
    virtual bool contains(const T& item) const = 0;

    /**
     * @brief Inserta un nuevo elemento garantizando la unicidad.
     * @param item Elemento a insertar.
     * @return true si la inserción fue exitosa, false si el elemento ya existía.
     */
    virtual bool add(const T& item) = 0;

    /**
     * @brief Extrae y retorna un elemento del conjunto.
     * @param item Elemento a eliminar.
     * @return El elemento eliminado (útil si T es un objeto complejo movible).
     * @throw std::invalid_argument Si el elemento no se encuentra en el conjunto.
     */
    virtual T remove(const T& item) = 0;
};



## El Modelo Secuencial (DynamicSet)

La implementación basada en arreglos dinámicos (`DynamicSet`) prioriza la eficiencia a nivel de microarquitectura del procesador. Al mantener los datos en un bloque de memoria contigua dentro del heap, esta estructura maximiza la localidad espacial. Durante las operaciones de escaneo para validar la unicidad, el *hardware prefetcher* de la CPU anticipa y carga secuencialmente las líneas de caché (L1/L2/L3), minimizando de forma drástica los fallos de caché (*cache misses*). 

Sin embargo, esta ventaja física impone una barrera teórica severa. Dado que cada inserción y eliminación exige un recorrido lineal completo, el costo operativo degenera rápidamente a $\mathcal{O}(N)$. En consecuencia, la carga secuencial masiva de elementos provoca un estancamiento computacional debido a su comportamiento asintótico global de $\mathcal{O}(N^2)$, lo que restringe el uso de esta arquitectura a conjuntos de tamaño reducido o estático.

In [2]:
/**
 * @class DynamicSet
 * @brief Implementación de ISet utilizando std::vector.
 * * Utiliza un arreglo dinámico interno. Ofrece un rendimiento óptimo en arquitecturas 
 * modernas para N pequeño/mediano debido al prefetching de hardware sobre memoria contigua, 
 * aunque presenta complejidad temporal O(N) para búsqueda, inserción y eliminación.
 * * @tparam T Tipo de los elementos almacenados.
 */
template <typename T>
class DynamicSet : public ISet<T> {
private:
    std::vector<T> data; ///< Contenedor interno privado para forzar el encapsulamiento y evitar mutaciones externas.

public:
    DynamicSet() = default;
    ~DynamicSet() override = default;

    void clear() override {
        data.clear();
        this->size = 0;
    }

    void print() const override {
        std::cout << "{ ";
        for (size_t i = 0; i < data.size(); ++i) {
            std::cout << data[i] << (i < data.size() - 1 ? ", " : " ");
        }
        std::cout << "}\n";
    }

    bool contains(const T& item) const override {
        // Se utiliza std::find para un escaneo lineal de memoria contigua.
        return std::find(data.begin(), data.end(), item) != data.end();
    }

    bool add(const T& item) override {
        if (contains(item)) return false; // Preservación de la invariante del conjunto (unicidad).
        
        data.push_back(item);
        this->size++;
        return true;
    }

    T remove(const T& item) override {
        auto it = std::find(data.begin(), data.end(), item);
        if (it != data.end()) {
            // Se utiliza std::move para evitar copias profundas innecesarias antes de la eliminación.
            T removedItem = std::move(*it);
            data.erase(it); // O(N) debido al desplazamiento de memoria requerido en vectores.
            this->size--;
            return removedItem;
        }
        throw std::invalid_argument("El elemento no existe.");
    }

    // ========================================================================
    // Operaciones entre Conjuntos
    // ========================================================================

    /**
     * @brief Computa la unión de este conjunto con otro (A ∪ B).
     * @param other Conjunto secundario.
     * @return Una nueva instancia de DynamicSet con los elementos combinados.
     */
    DynamicSet<T> unionWith(const DynamicSet<T>& other) const {
        DynamicSet<T> result;
        // Se asume complejidad O(N^2) en el peor caso debido a las validaciones de 'add'.
        for (const auto& item : this->data) result.add(item);
        for (const auto& item : other.data) result.add(item);
        return result;
    }

    /**
     * @brief Computa la intersección de este conjunto con otro (A ∩ B).
     * @param other Conjunto secundario.
     * @return Una nueva instancia con los elementos presentes en ambos conjuntos.
     */
    DynamicSet<T> intersectWith(const DynamicSet<T>& other) const {
        DynamicSet<T> result;
        for (const auto& item : this->data) {
            if (other.contains(item)) result.add(item);
        }
        return result;
    }

    /**
     * @brief Computa la diferencia asimétrica (A \ B).
     * @param other Conjunto a restar.
     * @return Nueva instancia con elementos exclusivos de este conjunto.
     */
    DynamicSet<T> differenceWith(const DynamicSet<T>& other) const {
        DynamicSet<T> result;
        for (const auto& item : this->data) {
            if (!other.contains(item)) result.add(item);
        }
        return result;
    }

    /**
     * @brief Evalúa si el conjunto actual es un subconjunto propio o impropio de otro (A ⊆ B).
     * @param other Conjunto a evaluar como superconjunto.
     * @return true si todos los elementos de este conjunto existen en el otro.
     */
    bool isSubsetOf(const DynamicSet<T>& other) const {
        // Optimización temprana: un subconjunto no puede tener mayor cardinalidad.
        if (this->getSize() > other.getSize()) return false; 
        
        for (const auto& item : this->data) {
            if (!other.contains(item)) return false;
        }
        return true;
    }
};

## Sistema de Validación y Trazabilidad de Errores

La verificación algorítmica prescinde del uso de aserciones estándar (`cassert`) para evitar la terminación abrupta del programa (`SIGABRT`) y el bloqueo del flujo de evaluación. En su lugar, se implementan macros personalizadas del preprocesador (`EXPECT_TRUE`, `EXPECT_EQUAL`) que capturan la expresión evaluada y su ubicación exacta en el código fuente mediante las directivas `#condition` y `__LINE__`. El encapsulamiento de estas directivas en bloques `do { ... } while(0)` garantiza la seguridad sintaxis durante la expansión del código. Este enfoque permite registrar de forma acumulativa los errores lógicos en un contador de fallos estático, habilitando la ejecución continua e ininterrumpida de toda la batería de pruebas.

La rutina de evaluación funcional somete la estructura a un escrutinio exhaustivo de sus invariantes matemáticas. Inicia comprobando el rechazo determinista de elementos duplicados para validar la unicidad. Posteriormente, certifica la correcta ejecución del álgebra de conjuntos, asegurando que las operaciones de unión, intersección, diferencia y subconjunto generen los resultados teóricos esperados sin mutar el estado de los conjuntos originales. El ciclo de pruebas concluye auditando las operaciones de mutación destructiva (extracción de elementos y vaciado de memoria), confirmando la coherencia final entre los datos alojados y la cardinalidad reportada por la estructura.

In [3]:
// ========================================================================
// Módulos de Validación y Benchmarking
// ========================================================================

/**
 * @brief Ejecuta pruebas unitarias mediante aserciones para verificar la correctitud algorítmica.
 */
#include <iostream>
#include <string>

// Macro para reportar errores sin detener la ejecución de todas las pruebas
#define EXPECT_TRUE(condition) \
    do { \
        if (!(condition)) { \
            std::cerr << "[FALLO] Linea " << __LINE__ \
                      << ": Evaluacion (" << #condition << ") no cumplida.\n"; \
            test_failures++; \
        } \
    } while(0)

#define EXPECT_EQUAL(expected, actual) \
    do { \
        if ((expected) != (actual)) { \
            std::cerr << "[FALLO] Linea " << __LINE__ \
                      << ": Se esperaba " << (expected) \
                      << ", pero se obtuvo " << (actual) << ".\n"; \
            test_failures++; \
        } \
    } while(0)

// Variable global estática para el conteo de errores en la unidad de traducción
static int test_failures = 0;

void runFunctionalTests() {
    DynamicSet<int> A;
    DynamicSet<int> B;

    // Poblar conjuntos iniciales y verificar unicidad
    EXPECT_TRUE(A.add(1));
    EXPECT_TRUE(A.add(2));
    EXPECT_TRUE(A.add(3));
    EXPECT_TRUE(A.add(3) == false); 

    B.add(3); B.add(4); B.add(5);

    // Validación de estado base
    EXPECT_EQUAL(3, A.getSize());
    EXPECT_TRUE(A.contains(2));

    // Aritmética: Unión
    DynamicSet<int> U = A.unionWith(B);
    EXPECT_EQUAL(5, U.getSize());
    EXPECT_TRUE(U.contains(1));
    EXPECT_TRUE(U.contains(5));

    // Aritmética: Intersección
    DynamicSet<int> I = A.intersectWith(B);
    EXPECT_EQUAL(1, I.getSize());
    EXPECT_TRUE(I.contains(3));

    // Aritmética: Diferencia
    DynamicSet<int> D = A.differenceWith(B);
    EXPECT_EQUAL(2, D.getSize());
    EXPECT_TRUE(D.contains(1));
    EXPECT_TRUE(D.contains(2));

    // Aritmética: Subconjunto
    DynamicSet<int> sub;
    sub.add(1); sub.add(2);
    EXPECT_TRUE(sub.isSubsetOf(A));
    EXPECT_TRUE(sub.isSubsetOf(B) == false);

    // Mutación y limpieza
    EXPECT_EQUAL(1, A.remove(1));
    EXPECT_EQUAL(2, A.getSize());
    
    A.clear();
    EXPECT_TRUE(A.isEmpty());

    // Reporte final del benchmark funcional
    if (test_failures == 0) {
        std::cout << "[OK] Pruebas funcionales de interfaz y algebra de conjuntos completadas (0 fallos).\n";
    } else {
        std::cerr << "[CRITICO] Se detectaron " << test_failures << " fallos durante las pruebas funcionales.\n";
    }
}
runFunctionalTests();

[OK] Pruebas funcionales de interfaz y algebra de conjuntos completadas (0 fallos).


## Metodología de Evaluación Empírica (Benchmarking)

El bloque de pruebas de rendimiento constituye un arnés empírico diseñado para validar el límite teórico de $\mathcal{O}(N^2)$ inherente a la arquitectura secuencial. La metodología de instrumentación aísla el costo algorítmico estricto de las operaciones del conjunto mediante la preasignación de memoria en el vector de prueba (`testData.reserve`), mitigando sesgos de reasignación dinámica en las métricas de tiempo.

La inyección de entropía se delega al motor de generación de números pseudoaleatorios *Mersenne Twister* (`std::mt19937`). Esta distribución uniforme de datos es crítica para someter a estrés los recorridos lineales de memoria, impidiendo que el *hardware prefetcher* anticipe patrones y distorsione la latencia real en las operaciones de búsqueda y mutación. 

La ejecución del perfilado mediante relojes de alta resolución (`std::chrono::high_resolution_clock`) a escalas de carga de trabajo incrementales ($10^3, 10^4, 5 \times 10^4$) expone el estrangulamiento del hardware. Mientras que el primer orden de magnitud se resuelve apalancando la localidad de la caché L1, el salto hacia las iteraciones superiores evidenciará un crecimiento cuadrático en la latencia, certificando físicamente la insostenibilidad de la verificación de unicidad polinómica para conjuntos de datos masivos.

In [4]:
/**
 * @brief Ejecuta perfiles de rendimiento temporal para operaciones críticas.
 * @param numElements Escala de la carga de trabajo para la prueba.
 */
void runBenchmark(size_t numElements) {
    DynamicSet<int> mySet;
    std::random_device rd;
    std::mt19937 gen(rd());
    std::uniform_int_distribution<> distrib(1, numElements * 2);

    std::vector<int> testData;
    testData.reserve(numElements);
    for (size_t i = 0; i < numElements; ++i) {
        testData.push_back(distrib(gen));
    }

    std::cout << "--- Benchmark: " << numElements << " elementos ---\n";

    auto start = std::chrono::high_resolution_clock::now();
    for (int val : testData) mySet.add(val);
    auto end = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double, std::milli> insertTime = end - start;
    std::cout << "Insercion O(N^2): " << std::fixed << std::setprecision(2) << insertTime.count() << " ms\n";

    start = std::chrono::high_resolution_clock::now();
    for (int val : testData) mySet.contains(val);
    end = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double, std::milli> searchTime = end - start;
    std::cout << "Busqueda O(N^2): " << searchTime.count() << " ms\n";

    start = std::chrono::high_resolution_clock::now();
    for (int val : testData) {
        try { mySet.remove(val); } catch (...) {}
    }
    end = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double, std::milli> removeTime = end - start;
    std::cout << "Eliminacion O(N^2): " << removeTime.count() << " ms\n\n";
}
    
runBenchmark(100);
runBenchmark(1000);
runBenchmark(10000);
runBenchmark(50000);


--- Benchmark: 100 elementos ---
Insercion O(N^2): 0.03 ms
Busqueda O(N^2): 0.03 ms
Eliminacion O(N^2): 0.29 ms

--- Benchmark: 1000 elementos ---
Insercion O(N^2): 2.20 ms
Busqueda O(N^2): 2.11 ms
Eliminacion O(N^2): 1.21 ms

--- Benchmark: 10000 elementos ---
Insercion O(N^2): 205.06 ms
Busqueda O(N^2): 204.61 ms
Eliminacion O(N^2): 39.50 ms

--- Benchmark: 50000 elementos ---
Insercion O(N^2): 5273.02 ms
Busqueda O(N^2): 5442.58 ms
Eliminacion O(N^2): 861.10 ms

